# EasyDE V2 — Exploratory Results Dashboard

This notebook reads all pipeline results across contrasts and strata, and produces:

1. **DE feature counts** — Boxplot of RUV-corrected DEG counts per contrast × cell type
2. **FGSEA heatmaps** — Top significant pathways per contrast × cell type
3. **Diagonal pathway heatmap** — Pathways clustered along the diagonal to reveal contrast/cell-type specificity

In [ ]:
# ===========================================================================
# Section 0 — Setup
# ===========================================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(pheatmap)
  library(scales)
  library(RColorBrewer)
})

# --- Project root (adjust if needed) ---
PROJECT_ROOT <- normalizePath(file.path(getwd(), ".."), mustWork = TRUE)
RESULTS_DIR  <- file.path(PROJECT_ROOT, "results")

cat("Project root:", PROJECT_ROOT, "\n")
cat("Results dir: ", RESULTS_DIR, "\n")

## Section 1 — Load All Contrast Summaries

In [ ]:
# Discover all contrast directories that have a summary/contrast_summary.csv
summary_files <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "contrast_summary.csv"))
cat(sprintf("Found %d contrast summary files:\n", length(summary_files)))
for (f in summary_files) cat("  ", basename(dirname(dirname(f))), "\n")

# Read and bind all summaries
deseq_all <- lapply(summary_files, function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  df
}) %>% bind_rows()

cat(sprintf("\nTotal rows: %d  |  Contrasts: %d  |  Strata: %d\n",
    nrow(deseq_all),
    n_distinct(deseq_all$contrast),
    n_distinct(deseq_all$biosample)))

# Quick preview
deseq_all %>%
  dplyr::select(biosample, contrast, status, n_degs_base, n_degs_ruv,
                n_pathways_base, n_pathways_ruv) %>%
  head(20)

## Section 2 — DE Features Boxplot (RUV-corrected)

In [ ]:
# Define contrast display order
contrast_order <- c(
  "Age", "Gender", "BMI",
  "Diabetic_vs_ND", "Prediabetic_vs_ND", "Prediabetic_vs_Diabetic",
  "ND_vs_T2D", "ND_vs_T1D",
  "HbA1c", "C_peptide",
  "AAB_ZNT8", "AAB_IAA", "AAB_IA2"
)

df_plot <- deseq_all %>%
  filter(status == "success") %>%
  mutate(
    n_degs_ruv = suppressWarnings(as.numeric(n_degs_ruv)),
    n_degs_ruv = tidyr::replace_na(n_degs_ruv, 0)
  ) %>%
  filter(contrast %in% contrast_order) %>%
  mutate(contrast = factor(contrast, levels = contrast_order))

cat(sprintf("Plotting %d successful contrast × strata combinations\n", nrow(df_plot)))

options(repr.plot.width = 14, repr.plot.height = 7)

ggplot(df_plot, aes(x = contrast, y = n_degs_ruv)) +
  geom_boxplot(aes(group = contrast), fill = "grey90", color = "grey50",
               outlier.shape = NA) +
  geom_point(aes(color = biosample),
             position = position_jitter(width = 0.18, height = 0),
             size = 2.6, alpha = 0.9) +
  scale_y_continuous(
    trans = scales::pseudo_log_trans(sigma = 1),
    breaks = c(0, 1, 10, 100, 1000, 10000),
    labels = scales::comma,
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    x = NULL,
    y = "Number of RUV DE features",
    color = "Cell type",
    title = "EasyDE V2 — RUV-corrected DE features per contrast"
  ) +
  theme_classic(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "right"
  )

## Section 3 — Load All FGSEA Results

In [ ]:
# Read merged fGSEA results from each contrast's summary folder
fgsea_ruv_files  <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "*_fgsea_ruv.tsv"))
fgsea_base_files <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "*_fgsea_base.tsv"))

read_fgsea <- function(path) {
  contrast_id <- basename(dirname(dirname(path)))
  df <- fread(path, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
  if (nrow(df) == 0) return(NULL)
  # Check for skip sentinels
  if ("skipped" %in% colnames(df)) return(NULL)
  df$contrast <- contrast_id
  df
}

fgsea_ruv <- lapply(fgsea_ruv_files, read_fgsea) %>%
  Filter(Negate(is.null), .) %>%
  bind_rows()

fgsea_base <- lapply(fgsea_base_files, read_fgsea) %>%
  Filter(Negate(is.null), .) %>%
  bind_rows()

cat(sprintf("fGSEA-RUV:  %d rows across %d contrasts, %d strata\n",
    nrow(fgsea_ruv), n_distinct(fgsea_ruv$contrast), n_distinct(fgsea_ruv$stratum)))
cat(sprintf("fGSEA-base: %d rows across %d contrasts, %d strata\n",
    nrow(fgsea_base), n_distinct(fgsea_base$contrast), n_distinct(fgsea_base$stratum)))

## Section 4 — FGSEA Heatmap: Top Pathways per Contrast × Cell Type

Filter for highly significant pathways, select top N per group using a composite score,
then display NES values as a heatmap.

In [ ]:
# --- Use RUV fGSEA results ---
gsea <- fgsea_ruv %>%
  filter(!is.na(padj), !is.na(NES), size >= 5)

# Keep pathways that are significant (padj < 0.05) in at least one stratum×contrast
sig_pathways <- gsea %>%
  group_by(pathway) %>%
  filter(min(padj) < 0.01) %>%
  ungroup()

# Score: -log10(padj) * |NES|, then pick top 5 per stratum×contrast
top_pathways <- sig_pathways %>%
  group_by(pathway) %>%
  mutate(nes_sd = sd(NES, na.rm = TRUE)) %>%
  ungroup() %>%
  filter(nes_sd >= 0.5) %>%
  group_by(stratum, contrast) %>%
  mutate(score = -log10(pmax(padj, 1e-300)) * abs(NES)) %>%
  slice_max(order_by = score, n = 5) %>%
  ungroup()

selected_paths <- unique(top_pathways$pathway)
cat(sprintf("Selected %d pathways with high variability and significance\n", length(selected_paths)))

In [ ]:
# Build NES matrix: pathway × (celltype:contrast)
heatmap_data <- sig_pathways %>%
  filter(pathway %in% selected_paths) %>%
  mutate(col_label = paste0(stratum, ":", contrast)) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
  tibble::column_to_rownames("pathway")

heatmap_data[is.na(heatmap_data)] <- 0

# Build significance-star matrix
sig_data <- sig_pathways %>%
  filter(pathway %in% selected_paths) %>%
  mutate(
    col_label = paste0(stratum, ":", contrast),
    star = case_when(
      padj < 0.001 ~ "***",
      padj < 0.01  ~ "**",
      padj < 0.05  ~ "*",
      TRUE         ~ ""
    )
  ) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "star", fill = "") %>%
  tibble::column_to_rownames("pathway")

# Align columns
sig_data <- sig_data[rownames(heatmap_data), colnames(heatmap_data)]
sig_data[is.na(sig_data)] <- ""

# Clean pathway names for display
clean_name <- function(x) {
  x %>%
    gsub("^(REACTOME_|KEGG_|HALLMARK_|GO_|WP_)", "", .) %>%
    gsub("_", " ", .) %>%
    substr(1, 60)
}
rownames(heatmap_data) <- clean_name(rownames(heatmap_data))
rownames(sig_data)     <- clean_name(rownames(sig_data))

# Draw heatmap
n_rows <- nrow(heatmap_data)
n_cols <- ncol(heatmap_data)
options(repr.plot.width  = max(12, n_cols * 0.6 + 6),
        repr.plot.height = max(8,  n_rows * 0.25 + 2))

pheatmap(
  heatmap_data,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  breaks = seq(-2, 2, length.out = 101),
  color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
  cellwidth = 18, cellheight = 14,
  border_color = "white",
  display_numbers = as.matrix(sig_data),
  number_color = "black",
  fontsize_number = 7,
  fontsize_row = 8,
  fontsize_col = 8,
  angle_col = 45,
  main = "fGSEA (RUV): Top variable significant pathways"
)

## Section 5 — Diagonal Heatmap: Contrast × Cell-Type Specificity

Permute rows so that pathways cluster along the diagonal, revealing which
pathways are specific to which contrast × cell-type combination.

**Method:** For each pathway (row), find which column has the maximum absolute NES.
Then sort rows by that column index. This produces a "diagonal" pattern where
pathway blocks align with their dominant contrast×celltype.

In [ ]:
# Use all significantly enriched pathways (padj < 0.05, |NES| >= 1)
diag_paths <- gsea %>%
  filter(padj < 0.05, abs(NES) >= 1) %>%
  group_by(stratum, contrast) %>%
  slice_max(order_by = abs(NES), n = 10) %>%
  ungroup() %>%
  pull(pathway) %>% unique()

rmat <- gsea %>%
  filter(pathway %in% diag_paths) %>%
  mutate(col_label = paste0(stratum, ":", contrast)) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
  tibble::column_to_rownames("pathway")

rmat[is.na(rmat)] <- 0

# --- Diagonal sort: order rows by the column with max |NES| ---
abs_rmat <- abs(as.matrix(rmat))
max_indices <- max.col(abs_rmat, ties.method = "first")
permutation_vector <- order(max_indices)
smat <- rmat[permutation_vector, ]

# Clean pathway names
rownames(smat) <- clean_name(rownames(smat))

n_rows <- nrow(smat)
n_cols <- ncol(smat)
options(repr.plot.width  = max(14, n_cols * 0.55 + 8),
        repr.plot.height = max(10, n_rows * 0.22 + 3))

pheatmap(
  smat,
  cluster_rows = FALSE,   # keep diagonal order
  cluster_cols = FALSE,
  breaks = seq(-2, 2, length.out = 101),
  color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
  cellwidth = 16, cellheight = 12,
  border_color = "white",
  fontsize_row = 7,
  fontsize_col = 8,
  angle_col = 45,
  main = "fGSEA (RUV): Diagonal — pathway specificity by contrast × cell type"
)

## Section 6 — FGSEA Heatmap: Cell-Type-Divergent Pathways

Find pathways that are enriched in **opposite directions** between cell types
for the same contrast (e.g., up in Beta, down in Alpha for Diabetic_vs_ND).
This reveals cell-type-specific biology.

In [ ]:
# For each contrast, find pathways with divergent NES across cell types
divergent_paths <- gsea %>%
  filter(padj < 0.05) %>%
  group_by(pathway, contrast) %>%
  filter(n() >= 2) %>%  # pathway must appear in ≥2 cell types
  summarise(
    has_pos = any(NES > 0),
    has_neg = any(NES < 0),
    nes_sd  = sd(NES),
    .groups = "drop"
  ) %>%
  filter(has_pos & has_neg, nes_sd >= 0.75) %>%
  pull(pathway) %>% unique()

cat(sprintf("%d pathways show divergent enrichment across cell types\n", length(divergent_paths)))

if (length(divergent_paths) >= 3) {
  div_mat <- gsea %>%
    filter(pathway %in% divergent_paths) %>%
    mutate(col_label = paste0(stratum, ":", contrast)) %>%
    reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
    tibble::column_to_rownames("pathway")

  div_mat[is.na(div_mat)] <- 0

  # Build padj star matrix
  div_stars <- gsea %>%
    filter(pathway %in% divergent_paths) %>%
    mutate(
      col_label = paste0(stratum, ":", contrast),
      star = case_when(
        padj < 0.001 ~ "***",
        padj < 0.01  ~ "**",
        padj < 0.05  ~ "*",
        TRUE         ~ ""
      )
    ) %>%
    reshape2::dcast(pathway ~ col_label, value.var = "star", fill = "") %>%
    tibble::column_to_rownames("pathway")

  div_stars <- div_stars[rownames(div_mat), colnames(div_mat)]
  div_stars[is.na(div_stars)] <- ""

  rownames(div_mat)   <- clean_name(rownames(div_mat))
  rownames(div_stars) <- clean_name(rownames(div_stars))

  n_rows <- nrow(div_mat)
  n_cols <- ncol(div_mat)
  options(repr.plot.width  = max(12, n_cols * 0.6 + 6),
          repr.plot.height = max(8,  n_rows * 0.25 + 2))

  pheatmap(
    div_mat,
    cluster_rows = TRUE,
    cluster_cols = TRUE,
    breaks = seq(-2, 2, length.out = 101),
    color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
    cellwidth = 18, cellheight = 14,
    border_color = "white",
    display_numbers = as.matrix(div_stars),
    number_color = "black",
    fontsize_number = 7,
    fontsize_row = 8,
    fontsize_col = 8,
    angle_col = 45,
    main = "Cell-type-divergent pathways (opposite NES across cell types)"
  )
} else {
  cat("Not enough divergent pathways found. Run more contrasts to populate this plot.\n")
}

## Section 7 — Pipeline Status Overview

In [ ]:
# Status tile plot: contrast × cell type
status_df <- deseq_all %>%
  mutate(
    status = factor(status, levels = c("success", "partial", "failed",
                                       "preflight_fail", "not_run")),
    contrast = factor(contrast, levels = rev(sort(unique(contrast)))),
    biosample = factor(biosample, levels = sort(unique(biosample)))
  )

options(repr.plot.width = 14, repr.plot.height = 8)

ggplot(status_df, aes(x = biosample, y = contrast, fill = status)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = ifelse(!is.na(n_degs_ruv), n_degs_ruv, "")),
            size = 2.5, color = "black") +
  scale_fill_manual(
    values = c(success = "#2ca02c", partial = "#ff7f0e",
               failed = "#d62728", preflight_fail = "#9467bd",
               not_run = "grey85"),
    drop = FALSE
  ) +
  labs(
    x = "Cell type", y = "Contrast", fill = "Status",
    title = "Pipeline completion status (numbers = RUV DEGs)"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank()
  )